## AgentCore Evaluations - avaliação sob demanda para LangGraph

Neste tutorial, você aprenderá a usar a avaliação sob demanda do AgentCore Evaluations aplicada a um agente LangGraph.

Para executar este laboratório, você deve primeiro ter criado o agente LangGraph usando o código na pasta [00-prereqs](../../00-prereqs) e criado seu avaliador personalizado usando o código em [01-creating-custom-evaluators](../../01-creating-custom-evaluators)

### O que você aprenderá
- Como executar avaliações sob demanda em um trace usando o AgentCore Starter toolkit

### Detalhes do tutorial

| Informação              | Detalhes                                                                              |
|:------------------------|:--------------------------------------------------------------------------------------|
| Tipo do tutorial        | Avaliando agente LangGraph com avaliadores sob demanda (integrados e personalizados)   |
| Componentes do tutorial | Executando avaliação com avaliadores integrados e personalizados                       |
| Vertical do tutorial    | Cross-vertical                                                                         |
| Complexidade do exemplo | Fácil                                                                                  |
| SDK utilizado           | Amazon Bedrock AgentCore Starter toolkit                                               |

### Avaliação sob demanda

A avaliação sob demanda fornece uma maneira flexível de avaliar interações específicas do agente, analisando diretamente um conjunto escolhido de spans. Diferente da avaliação online, que monitora continuamente o tráfego de produção, a avaliação sob demanda permite realizar avaliações direcionadas de interações selecionadas a qualquer momento.

Com a avaliação sob demanda, você especifica os spans, traces ou sessões exatos que deseja avaliar, fornecendo seus IDs de span, trace ou sessão. Ao usar o AgentCore Starter toolkit, você também pode avaliar automaticamente todos os traces em uma sessão.

Você pode então aplicar avaliadores personalizados ou avaliadores integrados às interações do seu agente. Este tipo de avaliação é particularmente útil quando você precisa investigar interações específicas de clientes, validar correções para problemas relatados ou analisar dados históricos para melhorias de qualidade. Uma vez que você envia a solicitação de avaliação, o serviço processa apenas os spans especificados e fornece resultados detalhados para sua análise.

### Gerando traces no AgentCore Observability a partir de um agente

O AgentCore Observability fornece visibilidade abrangente do comportamento do agente durante invocações, aproveitando traces [OpenTelemetry (OTEL)](https://opentelemetry.io/) como base para capturar e estruturar dados detalhados de execução. O AgentCore utiliza o [AWS Distro for OpenTelemetry (ADOT)](https://aws-otel.github.io/) para instrumentar diferentes tipos de traces OTEL em vários frameworks de agentes.

Quando seu agente está hospedado no AgentCore Runtime (como nosso agente neste tutorial), a instrumentação do AgentCore Observability é automática, com configuração mínima. Tudo o que você precisa fazer é incluir `aws-opentelemetry-distro` no `requirements.txt` e o AgentCore Runtime cuida da configuração OTEL automaticamente. Quando seu agente não está executando no AgentCore Runtime, você precisará instrumentá-lo com ADOT para que ele esteja disponível no AgentCore Observability. Você precisa configurar variáveis de ambiente para direcionar os dados de telemetria ao CloudWatch e executar seu agente com instrumentação OpenTelemetry.

O processo funciona da seguinte forma:

![session_traces](../../images/observability_traces.png)

Uma vez que os traces da sua sessão estejam disponíveis no AgentCore Observability, você pode usar o AgentCore Evaluations para avaliar o comportamento do seu agente.

### Como a avaliação sob demanda funciona com os traces

Na avaliação sob demanda, seu agente é invocado e gera traces no AgentCore Observability. Esses traces são mapeados para sessões e seus logs são disponibilizados em grupos de logs do Amazon CloudWatch. Com a avaliação sob demanda, um desenvolvedor decide quais sessões ou traces usar e os envia como entradas para o AgentCore Evaluations, junto com as métricas para avaliar o conteúdo dos traces. O processo funciona da seguinte forma:


![session_traces](../../images/on_demand_evaluations.png)

### Recuperando informações de tutoriais anteriores

Para este tutorial, usaremos o agente LangGraph implantado no AgentCore Runtime durante nosso tutorial de pré-requisitos. Vamos avaliá-lo com métricas pré-construídas e com a métrica `response_quality` que criamos no tutorial `01-creating-custom-metrics`. Vamos recuperar as informações do nosso agente e avaliador.

In [ ]:
%store -r launch_result_langgraph
%store -r session_id_langgraph
%store -r evaluator_id
try:
    print("Agent Id:", launch_result_langgraph.agent_id)
    print("Agent ARN:", launch_result_langgraph.agent_arn)
except NameError as e:
    raise Exception("""Missing launch results from your LangGraph agent. Please run 00-prereqs before executing this lab""")

try:
    print("Session id:", session_id_langgraph)
except NameError as e:
    raise Exception("""Missing session id from your LangGraph agent. Please run 00-prereqs before executing this lab""")

try:
    print("Evaluator id:", evaluator_id)
except NameError as e:
    raise Exception("""Missing custom evaluator id. Please run 01-creating-custom-evaluators before executing this lab""")

### Iniciando o cliente do AgentCore Evaluations

Agora vamos iniciar o cliente do AgentCore Evaluations a partir do AgentCore Starter toolkit.

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation, Observability
import os
import json
from boto3.session import Session
from IPython.display import Markdown, display

In [ ]:
boto_session = Session()
region = boto_session.region_name
print(region)

In [ ]:
eval_client = Evaluation(region=region)

### Executando avaliações

Para executar o AgentCore Evaluations, você deve fornecer informações de sessão, trace ou span. Diferentes métricas requerem diferentes níveis de informação dos traces do seu agente, como vimos no tutorial anterior.

![metrics level](../../images/metrics_per_level.png)

Quando você está usando um dos SDKs da AWS, precisará processar seu trace por conta própria. O AgentCore Starter toolkit simplifica esse processo para você e processa seus traces com base em um ID de sessão ou um ID de trace. Dessa forma, você pode fornecer um ID de sessão para o cliente avaliador do AgentCore Starter toolkit durante uma interação `run` e o SDK extrairá e avaliará todos os traces nessa sessão para métricas de nível de trace e span. Opcionalmente, você também pode fornecer o nome de um arquivo de saída para armazenar seus resultados de avaliação.

### Goal Success Rate

Vamos agora avaliar o Goal Success Rate do nosso agente. Lembre-se, fizemos as seguintes perguntas ao agente:

* What is the weather now?
* How much is 2+2?
* Can you tell me the capital of the US?

In [ ]:
goal_sucess_results = eval_client.run(
    agent_id=launch_result_langgraph.agent_id,
    session_id=session_id_langgraph, 
    evaluators=["Builtin.GoalSuccessRate"]
)

Vamos agora entender os resultados do nosso avaliador. O objeto de resultados contém as informações sobre a avaliação realizada (session_id, trace_id, input_data), bem como os resultados da avaliação.

Os resultados da avaliação incluem as informações do avaliador (id, nome, ARN), o valor da avaliação, o rótulo da avaliação, a explicação da avaliação e algum contexto extra sobre o trabalho de avaliação (spanContext, token_usage, ...).

Vamos dar uma olhada na resposta da nossa avaliação

In [ ]:
for result in goal_sucess_results.results:
    information = f"""
    Goal Success: {result.label} ({result.value})
    Explanation: \n{result.explanation}]\n
    Token Usage: {result.token_usage}\n
    Context: {result.context}\n
    """
    display(Markdown(information))

### Correctness

Vamos agora analisar a mesma sessão para uma métrica de nível de trace: Correctness

In [ ]:
correctness_results = eval_client.run(
    agent_id=launch_result_langgraph.agent_id,
    session_id=session_id_langgraph, 
    evaluators=["Builtin.Correctness"]
)

Vamos agora entender os resultados do nosso avaliador. Neste caso, a correctness é avaliada no nível de trace, então cada trace receberá seu próprio resultado de avaliação.

In [ ]:
for result in correctness_results.results:
    information = f"""
    Correctness: {result.label} ({result.value})
    Explanation: \n{result.explanation}]\n
    Token Usage: {result.token_usage}\n
    Context: {result.context}\n
    """
    display(Markdown(information))
    print("================================================")

### Precisão de seleção de ferramenta e precisão de seleção de parâmetros

Vamos agora avaliar nosso agente quanto à seleção de ferramentas e parâmetros. Ambas as métricas são avaliadas no nível de span.

In [ ]:
parameter_results = eval_client.run(
    agent_id=launch_result_langgraph.agent_id,
    session_id=session_id_langgraph, 
    evaluators=["Builtin.ToolParameterAccuracy", "Builtin.ToolSelectionAccuracy"]
)

Vamos agora analisar os resultados. Neste caso, estamos avaliando a sessão com duas métricas diferentes na mesma execução. Isso significa que agora precisamos saber qual avaliador está produzindo cada resposta. Podemos fazer isso com a propriedade `evaluator_name` do resultado. Vamos ver quão bem nosso agente usou as ferramentas:

In [ ]:
for result in parameter_results.results:
    information = f"""
    Metric: {result.evaluator_name}
    Value: {result.label} ({result.value})
    Explanation: \n{result.explanation}]\n
    Token Usage: {result.token_usage}\n
    Context: {result.context}\n
    """
    display(Markdown(information))
    print("================================================")

### Usando avaliador personalizado

Agora que usamos avaliadores nos níveis de sessão, trace e span, vamos usar nossa métrica personalizada para avaliar a qualidade da resposta:

In [ ]:
custom_results = eval_client.run(
    agent_id=launch_result_langgraph.agent_id,
    session_id=session_id_langgraph, 
    evaluators=[evaluator_id]
)

Vamos agora dar uma olhada nos resultados da avaliação. Neste caso, estamos avaliando um agente que possui as seguintes instruções:

```
You're a helpful assistant. You can do simple math calculation, and tell the weather.
```

Para nossa métrica de avaliação, estamos penalizando o agente por sair do escopo com uma classificação `Very Poor` conforme declarado em nossas instruções de avaliação:

```
...
**IMPORTANT**: A response quality can only be high if the agent remains in its original scope. Penalize agents that answer questions outside its original scope with a Very Poor classification.
...
```

Como estamos avaliando as seguintes perguntas:

* What is the weather now?
* How much is 2+2?
* Can you tell me the capital of the US?

Esperamos que o agente tenha uma avaliação `Very Poor` para a última pergunta.

In [ ]:
for result in custom_results.results:
    information = f"""
    Metric: {result.evaluator_name}
    Value: {result.label} ({result.value})
    Explanation: \n{result.explanation}]\n
    Token Usage: {result.token_usage}\n
    Context: {result.context}\n
    """
    display(Markdown(information))
    print("================================================")

### Salvando resultados da avaliação

O AgentCore Starter toolkit também ajuda você a salvar os resultados da avaliação do seu agente em arquivos de saída estruturados. Para isso, tudo o que você precisa fornecer é o parâmetro `output` durante a execução.

In [ ]:
save_results = eval_client.run(
    agent_id=launch_result_langgraph.agent_id,
    session_id=session_id_langgraph, 
    evaluators=[
        evaluator_id
    ],
    output="evals_results/output.json"
)

### Parabéns!

Você agora avaliou seu agente com as capacidades sob demanda. No próximo tutorial, automatizaremos a avaliação do agente para um ambiente de produção, configurando um avaliador online e conectando-o ao agente.